# Swin V2 y el token [CLS] de ViT

Notebook opcional del **módulo 11** del curso (en la app: `uv run dev` y la ruta `#/modulo/vision-transformers`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

ds = load_dataset("Matthijs/snacks")
etiquetas = ds["train"].features["label"].names
print(etiquetas)

In [ ]:
nombre = "microsoft/swinv2-tiny-patch4-window8-256"
procesador = AutoImageProcessor.from_pretrained(nombre)
modelo = AutoModelForImageClassification.from_pretrained(nombre, num_labels=len(etiquetas), ignore_mismatched_sizes=True)
x = procesador(ds["train"][0]["image"], return_tensors="pt")
print(modelo(**x).logits.shape)

Para el ajuste fino completo, usa `Trainer` con `TrainingArguments(learning_rate=5e-5, num_train_epochs=3, per_device_train_batch_size=16)` y aumento de datos (`torchvision.transforms`). Con GPU tarda minutos.

In [ ]:
from transformers import ViTModel

vit = ViTModel.from_pretrained("google/vit-base-patch16-224", output_attentions=True, attn_implementation="eager")
vp = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
with torch.no_grad():
    att = vit(**vp(ds["train"][0]["image"], return_tensors="pt")).attentions[-1][0].mean(0)
mapa = att[0, 1:].reshape(14, 14)        # atención del token [CLS] a los 196 parches
import matplotlib.pyplot as plt
plt.imshow(mapa)
plt.title("Atención del token [CLS] (última capa)")
plt.show()